# What's Hot in Tech in 2026? A GitHub Trending Analysis

In this project you'll collect **live data** about the most-starred GitHub repositories created in the last 12 months and use it to answer:

| Part | Question | New skills (beyond the tips project) |
|---|---|---|
| **0. Collect** | How do we get the data? | Calling an API, JSON, pagination, rate limits |
| **1. Clean** | Is the data usable? | Flattening nested data, missing values, dates, lists inside cells |
| **2. Describe** | What do typical repos look like? | Very skewed data, **log scales** |
| **3. Languages** | Which programming languages dominate? | Ranking categories, "count vs typical value" |
| **4. Topics & AI** | What are people building? How big is AI? | **Exploding** lists, keyword flags |
| **5. Momentum** | What's growing fastest? | Rates over time, **time bias** |
| **6. Stars vs forks** | Do popular repos get copied more? | Log-log regression, Spearman correlation |
| **7. Conclude** | What did we find? | Writing up live results |

**How to use this notebook**
- Upload it to [Google Colab](https://colab.research.google.com) (*File → Upload notebook*) and run the cells **in order** with Shift + Enter.
- Read the explanation **before** each cell and the "What to notice" notes **after** it.
- ✏️ **Try it yourself** cells are exercises. Answers are hidden under "Show answer".

> **Important:** because the data is live, your exact numbers will differ from anyone else's (and from the day you run it next). That's normal. The notes tell you *what kind of pattern* to look for, and the code prints the real numbers.

---
# Part 0: Collect the data

## Step 0.1: Import the libraries

Same four as the tips project, plus:
- **requests**: sends requests to websites and APIs from Python.
- **time** and **datetime**: for pausing between requests and working with dates.

In [1]:
import time
from datetime import datetime, timedelta, timezone

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 2)
pd.set_option("display.max_colwidth", 60)

BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#8a8985"   # colours used in our charts

ModuleNotFoundError: No module named 'requests'

## Step 0.2: What is an API?

An **API** (Application Programming Interface) is a door a website opens for programs instead of people. You send a URL with your question, and it sends back **data** instead of a web page.

We'll use GitHub's **search API**. Our question in plain English:

> "Give me repositories **created in the last 365 days** with **more than 50 stars**, sorted by stars, highest first."

In GitHub's search language that's: `created:>2025-10-27 stars:>50` (the date is calculated automatically below).

Three rules of this API you need to know:
1. **Pagination**: it returns at most **100 results per request** (a "page"), so we ask for page 1, page 2, ... page 10.
2. **1,000 result cap**: search never returns more than 1,000 results in total, so 10 pages is the maximum.
3. **Rate limit**: without logging in, you can make about **10 search requests per minute**. So we pause ~7 seconds between pages. The whole download takes **about 1–2 minutes**.

In [ ]:
since = (datetime.now(timezone.utc) - timedelta(days=365)).strftime("%Y-%m-%d")
query = f"created:>{since} stars:>50"
print("Search query:", query)

## Step 0.3: Look at ONE page first

Before downloading everything, always look at a small sample. The API answers in **JSON**, which Python turns into a **dictionary** (`{key: value}`) containing lists of more dictionaries.

- `status_code` 200 means success. 403 or 429 means "slow down" (rate limit).
- `total_count` is how many repos match in total.
- `items` is the list of repos on this page.

*(Optional: if you have a GitHub account, you can paste a personal access token between the quotes below to get higher rate limits. Leaving it empty works fine.)*

In [ ]:
GITHUB_TOKEN = ""   # optional, leave empty

headers = {"Accept": "application/vnd.github+json"}
if GITHUB_TOKEN:
    headers["Authorization"] = f"Bearer {GITHUB_TOKEN}"

URL = "https://api.github.com/search/repositories"
params = {"q": query, "sort": "stars", "order": "desc", "per_page": 100, "page": 1}

response = requests.get(URL, params=params, headers=headers, timeout=30)
print("Status code:", response.status_code)

data = response.json()
print("Total matching repos:", data.get("total_count"))
print("Repos on this page:", len(data.get("items", [])))

Let's peek inside the first repo. It has **dozens of fields**; we print just a few.

In [ ]:
first = data["items"][0]
for key in ["full_name", "description", "language", "stargazers_count",
            "forks_count", "topics", "created_at", "license", "owner"]:
    value = first.get(key)
    if key == "owner":
        value = {"login": value["login"], "type": value["type"]}
    print(f"{key:18} → {value}")

**What to notice:**
- Most fields are simple values (text, numbers).
- `topics` is a **list** inside a single field, e.g. `['ai', 'llm', 'agents']`.
- `license` and `owner` are **dictionaries inside the dictionary** (nested data). A table can't hold that directly, so we'll pull out just the parts we need in Part 1.
- `created_at` is a date written as text, e.g. `2026-03-14T09:21:55Z`. We'll convert it to a real date.

## Step 0.4: Download all 10 pages

Now we wrap the request in a **function** that:
- loops through pages 1–10,
- waits between pages to respect the rate limit,
- if GitHub says "too many requests", waits a minute and tries again (up to 3 times),
- stops early if a page comes back empty.

Run it and wait for it to finish (~1–2 minutes). It prints progress as it goes.

In [ ]:
def fetch_repos(query, pages=10, pause=7):
    all_items = []
    for page in range(1, pages + 1):
        params = {"q": query, "sort": "stars", "order": "desc", "per_page": 100, "page": page}
        for attempt in range(3):
            r = requests.get(URL, params=params, headers=headers, timeout=30)
            if r.status_code == 200:
                break
            print(f"  page {page}: status {r.status_code}, waiting 60s and retrying...")
            time.sleep(60)
        else:
            print("Giving up after 3 tries. Run this cell again in a few minutes.")
            break

        items = r.json().get("items", [])
        if not items:
            break
        all_items.extend(items)
        print(f"page {page:2}: got {len(items)} repos (total so far: {len(all_items)})")
        time.sleep(pause)
    return all_items

raw_items = fetch_repos(query)
print("\nDownloaded", len(raw_items), "repositories")

---
# Part 1: Clean the data

## Step 1.1: Turn the JSON into a table

We build one small dictionary per repo with only the fields we want, then hand the list to `pd.DataFrame`. This is where we **flatten** the nested parts:
- `license` might be missing (`None`), so we write `(repo.get("license") or {}).get("spdx_id")`. The `or {}` means "if there's no licence, use an empty dictionary instead", which avoids an error.
- `owner` → we keep the `type`, which is either **User** (a person) or **Organization** (a company/group).

In [ ]:
rows = []
for repo in raw_items:
    rows.append({
        "name":        repo["full_name"],
        "description": repo.get("description"),
        "language":    repo.get("language"),
        "stars":       repo["stargazers_count"],
        "forks":       repo["forks_count"],
        "open_issues": repo["open_issues_count"],
        "topics":      repo.get("topics", []),
        "license":     (repo.get("license") or {}).get("spdx_id"),
        "owner_type":  repo["owner"]["type"],
        "created_at":  repo["created_at"],
        "pushed_at":   repo["pushed_at"],
        "url":         repo["html_url"],
    })

df = pd.DataFrame(rows)
print(df.shape)
df.head()

## Step 1.2: Check for missing values and duplicates

Unlike the tips data, real data has gaps. `isna().sum()` counts missing values per column. We also check for **duplicate** repos: search results can shift slightly between pages while you're downloading, so the same repo can appear twice.

In [ ]:
print("Missing values per column:")
print(df.isna().sum())
print()
print("Duplicate repos:", df.duplicated(subset="name").sum())

**What to notice:**
- `language` is often missing: GitHub couldn't detect one (e.g. repos that are mostly documentation, prompts or lists of links).
- `description` and `license` are missing for some repos too.

**Decisions** (always write down your cleaning decisions):
- Missing language → label it `"None/Docs"` rather than deleting those rows. They're real repos, often "awesome-lists" and prompt collections, and dropping them would hide something interesting.
- Missing license → `"No license"`.
- Missing description → empty text `""`, so text searches don't break.
- Duplicates → drop them.

In [ ]:
df = df.drop_duplicates(subset="name")
df["language"] = df["language"].fillna("None/Docs")
df["license"] = df["license"].fillna("No license")
df["description"] = df["description"].fillna("")

print(df.isna().sum().sum(), "missing values left")
print(len(df), "repos after removing duplicates")

## Step 1.3: Convert dates and create new columns

`pd.to_datetime` converts date text into real dates you can do maths with. Then we create (feature engineering, like `tip_pct` in the tips project):
- **age_days**: how many days old the repo is.
- **stars_per_day**: stars ÷ age, a measure of **momentum**. A repo with 20,000 stars in 30 days is hotter than one with 30,000 stars in 300 days.
- **days_since_push**: days since the last code change (is it still active?).
- **created_month**: the month it was created, for trends.

In [ ]:
now = pd.Timestamp.now(tz="UTC")
df["created_at"] = pd.to_datetime(df["created_at"])
df["pushed_at"]  = pd.to_datetime(df["pushed_at"])

df["age_days"]        = (now - df["created_at"]).dt.days.clip(lower=1)   # clip avoids dividing by 0
df["stars_per_day"]   = df["stars"] / df["age_days"]
df["days_since_push"] = (now - df["pushed_at"]).dt.days
df["created_month"]   = df["created_at"].dt.tz_localize(None).dt.to_period("M")

df[["name", "stars", "age_days", "stars_per_day", "days_since_push", "created_month"]].head()

## Step 1.4: Save a copy

Save the cleaned data to a CSV file. Since the data is live, this is your **snapshot**: if you re-run the notebook next week, you'll get different numbers, so saving lets you reproduce your results. (In Colab, the file appears in the 📁 Files panel on the left; right-click → Download.)

In [ ]:
snapshot_date = now.strftime("%Y-%m-%d")
df.to_csv(f"github_hot_repos_{snapshot_date}.csv", index=False)
print("Saved", f"github_hot_repos_{snapshot_date}.csv")

### ✏️ Try it yourself

1. How many repos are owned by an **Organization** vs a **User**? (Hint: `value_counts()`)
2. What are the 5 most common licenses?

<details><summary>Show answer</summary>

```python
df["owner_type"].value_counts()
df["license"].value_counts().head(5)
```
Expect MIT and Apache-2.0 to be the most common licenses, with a sizeable "No license" group.
</details>

In [ ]:
# Write your code here

---
# Part 2: Describe the data

## Step 2.1: Summary statistics

In [ ]:
df[["stars", "forks", "open_issues", "age_days", "stars_per_day"]].describe()

**What to notice:** compare the **mean** with the **median (50%)** for stars. The mean will be **much** larger, several times the median. In the tips data, the mean bill was only a little above the median. Here, a few mega-popular repos pull the average way up.

This is called a **heavily skewed** (or "long-tail") distribution, and it's typical of anything about popularity: YouTube views, app downloads, follower counts. **For skewed data, the median is the better "typical" value.**

## Step 2.2: Why we need a log scale

Let's plot the stars distribution two ways:
- **Left, normal scale:** each step on the axis adds the same amount (0, 10k, 20k...).
- **Right, log scale:** each step **multiplies** by 10 (100, 1,000, 10,000...). This spreads out the small values and squeezes the huge ones, so you can see the whole shape.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

sns.histplot(df["stars"], bins=40, ax=axes[0], color=BLUE)
axes[0].set_title("Stars (normal scale)")
axes[0].set_xlabel("Stars")

sns.histplot(df["stars"], bins=40, log_scale=True, ax=axes[1], color=BLUE)
axes[1].set_title("Stars (log scale)")
axes[1].set_xlabel("Stars (log scale)")

plt.tight_layout()
plt.show()

**What to notice:** on the left, almost everything is crammed into one bar at the far left, with a long empty tail. The chart is useless. On the right (log scale), you can actually see how repos are spread out. **Whenever data spans several orders of magnitude (10s to 100,000s), reach for a log scale.**

## Step 2.3: The top 15 repos

`nlargest(15, "stars")` gives the most-starred repos. A horizontal bar chart is best for ranked items with long names: the names stay readable.

In [ ]:
top15 = df.nlargest(15, "stars")
display(top15[["name", "language", "stars", "age_days", "description"]])

plt.figure(figsize=(10, 6))
sns.barplot(data=top15, y="name", x="stars", color=BLUE)
plt.title("15 most-starred repos created in the last 12 months")
plt.xlabel("Stars")
plt.ylabel("")
plt.show()

**What to notice:** read the descriptions. What kinds of projects are at the very top? Write down 2–3 themes you see (for example AI agents, developer tools, or a viral app). You'll test whether those themes hold across all 1,000 repos in Part 4.

---
# Part 3: Which languages dominate?

## Step 3.1: Count repos per language

In [ ]:
lang_counts = df["language"].value_counts()
print(lang_counts.head(15))
print()
print("Number of different languages:", df["language"].nunique())

In [ ]:
top_langs = lang_counts.head(12)

plt.figure(figsize=(10, 5.5))
sns.barplot(x=top_langs.values, y=top_langs.index, color=BLUE)
plt.title("Number of hot repos by language (top 12)")
plt.xlabel("Number of repos")
plt.ylabel("")
plt.show()

**What to notice:** you'll most likely see **Python** and **TypeScript** far ahead of everything else, with Go, Rust and JavaScript behind. Think about why: Python is the main language of AI/ML, and TypeScript is the main language of web apps and many AI developer tools.

Also look at where **None/Docs** ranks. A big number there means many hot repos aren't code at all (prompt collections, guides, lists).

## Step 3.2: Count vs typical popularity

"Most repos" isn't the same as "most popular repos". Let's compute, for each language, the **count** and the **median stars**. We use `agg` with named results, which is cleaner than in the tips project.

We only include languages with **at least 10 repos**. Remember the lesson from tips: averages built on a handful of rows are unreliable.

In [ ]:
lang_stats = (
    df.groupby("language")
      .agg(repos=("name", "count"),
           median_stars=("stars", "median"),
           total_stars=("stars", "sum"))
      .query("repos >= 10")
      .sort_values("repos", ascending=False)
)
lang_stats

**What to notice:** the language with the **most** repos isn't necessarily the one with the **highest median stars**. A smaller language might have fewer repos that are each more popular. When you compare groups, always ask: *"Am I looking at how many, or how big?"* They can tell different stories.

## Step 3.3: Spread of stars by language (box plot on a log scale)

Like the tips box plots, but with a log axis because stars are so skewed. We show the 8 most common languages.

In [ ]:
langs8 = lang_counts.head(8).index
subset = df[df["language"].isin(langs8)]

plt.figure(figsize=(11, 5))
sns.boxplot(data=subset, x="language", y="stars", order=langs8, color=BLUE, log_scale=True)
plt.title("Stars by language (log scale), 8 most common languages")
plt.xlabel("")
plt.ylabel("Stars (log scale)")
plt.show()

**What to notice:** the boxes probably **overlap a lot**. Language alone doesn't decide how popular a repo gets. The dots far above each box are the breakout hits.

### ✏️ Try it yourself

Which language has the highest **median stars_per_day** (momentum), among languages with at least 10 repos?

<details><summary>Show answer</summary>

```python
(df.groupby("language")
   .agg(repos=("name", "count"), median_momentum=("stars_per_day", "median"))
   .query("repos >= 10")
   .sort_values("median_momentum", ascending=False))
```
</details>

In [ ]:
# Write your code here

---
# Part 4: Topics, and how big is AI?

## Step 4.1: Explode the topics

Each repo's `topics` cell holds a **list**, like `['ai', 'llm', 'agents']`. To count topics we need **one row per topic**. `explode()` does exactly that: a repo with 3 topics becomes 3 rows.

Here's a tiny example first so you can see what it does:

In [ ]:
example = pd.DataFrame({"repo": ["A", "B"], "topics": [["ai", "llm"], ["rust"]]})
print("Before explode:")
print(example)
print("\nAfter explode:")
print(example.explode("topics"))

In [ ]:
topics = df.explode("topics")["topics"].dropna()
print("Repos with at least one topic:", (df["topics"].str.len() > 0).sum(), "of", len(df))
print()
top_topics = topics.value_counts().head(20)
top_topics

In [ ]:
plt.figure(figsize=(10, 7))
sns.barplot(x=top_topics.values, y=top_topics.index, color=BLUE)
plt.title("20 most common topics among hot repos")
plt.xlabel("Number of repos")
plt.ylabel("")
plt.show()

**What to notice:** expect the list to be dominated by AI words: things like `ai`, `llm`, `agents`/`ai-agents`, `mcp` (Model Context Protocol, a standard for connecting AI tools), `claude`, `openai`, `rag`. Look for any **non-AI** topics that still make the top 20. Those are the other hot areas.

⚠️ Topics are added by repo owners, so they're inconsistent (`ai-agents` vs `agents` vs `agent`) and some repos have none. That's why the next step also checks descriptions.

## Step 4.2: Flag AI repos

We'll mark a repo as **AI-related** if its topics *or* description contain an AI keyword. This is a **keyword flag**, a simple but very common technique.

- We join topics and description into one lowercase string per repo.
- `str.contains(pattern)` checks each string against a **regular expression** (regex), a pattern language for text. `\b` means "word boundary" and `(?: ... | ... )` means "any one of these words", so `ai` matches "ai" but not "said" or "email".
- The result is a True/False column. The **mean** of a True/False column is the share of True values (True counts as 1, False as 0).

In [ ]:
AI_PATTERN = (r"\b(?:ai|llms?|gpt|agents?|agentic|mcp|rag|claude|openai|gemini|deepseek|"
              r"machine[- ]learning|deep[- ]learning|neural|transformers?|diffusion|"
              r"chatbot|copilot|genai|generative|prompts?|ollama)\b")

text = (df["topics"].apply(" ".join) + " " + df["description"]).str.lower()
df["is_ai"] = text.str.contains(AI_PATTERN, regex=True)

print(f"AI-related repos: {df['is_ai'].sum()} of {len(df)}  ({df['is_ai'].mean()*100:.0f}%)")

Always **spot-check** a flag like this. Look at a few repos it marked as AI and a few it didn't, and ask "did it get these right?"

In [ ]:
print("Marked as AI:")
display(df[df["is_ai"]].sample(5, random_state=1)[["name", "description"]])
print("Marked as NOT AI:")
display(df[~df["is_ai"]].sample(5, random_state=1)[["name", "description"]])

**What to notice:** you'll probably find a few mistakes in each direction. That's normal. Write down your flag's limits in your conclusions. If you spot a word that's clearly missing (or causing false matches), edit `AI_PATTERN` and re-run the cell.

## Step 4.3: Do AI repos get more stars?

In [ ]:
ai_compare = df.groupby("is_ai").agg(
    repos=("name", "count"),
    median_stars=("stars", "median"),
    median_stars_per_day=("stars_per_day", "median"),
)
ai_compare.index = ai_compare.index.map({True: "AI", False: "Not AI"})
ai_compare

In [ ]:
plt.figure(figsize=(7, 5))
plot_df = df.assign(group=df["is_ai"].map({True: "AI", False: "Not AI"}))
sns.boxplot(data=plot_df, x="group", y="stars", order=["AI", "Not AI"],
            palette=[ORANGE, BLUE], hue="group", legend=False, log_scale=True)
plt.title("Stars: AI vs non-AI repos (log scale)")
plt.xlabel("")
plt.ylabel("Stars (log scale)")
plt.show()

**What to notice:** compare **two** things. First, the **share** of hot repos that are AI (Step 4.2), which tells you how much of the attention AI takes up. Second, the **median stars** of AI vs non-AI, which tells you whether an AI repo is *individually* more popular. AI could dominate by sheer numbers without each AI repo being more popular, or both could be true.

### ✏️ Try it yourself

Which **language** has the highest share of AI repos (among languages with at least 10 repos)?

<details><summary>Show answer</summary>

```python
(df.groupby("language")
   .agg(repos=("name", "count"), ai_share=("is_ai", "mean"))
   .query("repos >= 10")
   .sort_values("ai_share", ascending=False))
```
</details>

In [ ]:
# Write your code here

---
# Part 5: Momentum: what's growing fastest?

## Step 5.1: Fastest-growing repos

Total stars favour older repos, because they've had more time. `stars_per_day` levels the field.

In [ ]:
fastest = df.nlargest(15, "stars_per_day")
fastest[["name", "language", "stars", "age_days", "stars_per_day", "is_ai"]]

**What to notice:** compare this list with the top-15-by-stars list from Step 2.3. Repos that appear here but **not** there are young and rising quickly. These are the best answer to "what's hot *right now*". How many of them are AI?

## Step 5.2: When were the hot repos created? (and a trap)

Let's count how many of our hot repos were created in each month.

In [ ]:
monthly = df.groupby("created_month").size()

plt.figure(figsize=(11, 4.5))
plt.plot(monthly.index.astype(str), monthly.values, marker="o", color=BLUE, linewidth=2)
plt.title("Hot repos by month created")
plt.ylabel("Number of repos in our top 1,000")
plt.xticks(rotation=45)
plt.show()

**⚠️ The trap:** you'll probably see **fewer** repos from the most recent months. Does that mean less exciting stuff is being built lately? **No!** Our dataset only includes the 1,000 most-starred repos. Repos created last month have had only a few weeks to collect stars, so fewer of them make the cut.

This is called **time bias** (a close cousin of **selection bias**): *how you chose the data* creates a pattern that isn't really about the world. Spotting this kind of trap is one of the most valuable skills an analyst has. Put it in your conclusions.

**A fairer comparison** uses momentum instead of counts:

In [ ]:
monthly_momentum = df.groupby("created_month")["stars_per_day"].median()

plt.figure(figsize=(11, 4.5))
plt.plot(monthly_momentum.index.astype(str), monthly_momentum.values, marker="o", color=BLUE, linewidth=2)
plt.title("Median stars per day, by month created")
plt.ylabel("Median stars per day")
plt.xticks(rotation=45)
plt.show()

**What to notice:** recent months usually show **higher** stars per day. That's partly real (new repos get a burst of attention at launch) and partly the same selection effect in reverse: a very new repo only makes our list if it's growing *extremely* fast. Neither chart alone gives the full truth, so describe both in your write-up.

## Step 5.3: Is it still being maintained?

`days_since_push` tells us how recently code was changed. A hot repo that hasn't been touched in months may have been a one-time viral hit.

In [ ]:
active = (df["days_since_push"] <= 30).mean() * 100
print(f"{active:.0f}% of hot repos had code pushed in the last 30 days")

sns.histplot(df["days_since_push"], bins=30, color=BLUE)
plt.title("Days since last code push")
plt.xlabel("Days")
plt.show()

---
# Part 6: Stars vs forks: regression on a log scale

A **star** is like a bookmark ("I like this"). A **fork** is a copy someone makes to change or build on the code. Do more popular repos get copied more?

## Step 6.1: Scatter plot, normal vs log-log

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(data=df, x="stars", y="forks", ax=axes[0], color=BLUE, alpha=0.5, s=20)
axes[0].set_title("Stars vs forks (normal scale)")

sns.scatterplot(data=df, x="stars", y="forks", ax=axes[1], color=BLUE, alpha=0.5, s=20)
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Stars vs forks (log-log scale)")

plt.tight_layout()
plt.show()

**What to notice:** on the normal scale, nearly every dot is crushed into the bottom-left corner. On the **log-log** scale, the cloud spreads out and you can see a clear upward trend. (`alpha=0.5` makes dots semi-transparent so overlapping ones show up darker.)

*Note: repos with 0 forks can't be shown on a log scale (log of 0 is undefined), so a few points may be missing from the right chart.*

## Step 6.2: Two kinds of correlation

- **Pearson** correlation (what we used in the tips project) measures how well points fit a **straight line**. A few giant repos can distort it badly.
- **Spearman** correlation first converts each value to its **rank** (1st, 2nd, 3rd...) and then compares ranks. It asks: "do repos with more stars *tend* to have more forks?" Giant outliers can't distort it, so it's better for skewed data.

In [ ]:
pearson = df["stars"].corr(df["forks"], method="pearson")
spearman = df["stars"].corr(df["forks"], method="spearman")
print(f"Pearson  r = {pearson:.2f}")
print(f"Spearman ρ = {spearman:.2f}")

**What to notice:** if the two numbers differ a lot, outliers are influencing Pearson. Spearman is the one to trust and report here.

## Step 6.3: Fit a line on the log scale

On log-log axes the relationship looks like a straight line, so we fit a regression line to `log10(stars)` and `log10(forks)` using `np.polyfit`, as in the tips project. We drop 0-fork repos first.

**How to read the slope on a log-log scale:**
- slope ≈ 1 → forks grow **in proportion** to stars (10× the stars → 10× the forks)
- slope < 1 → forks grow **slower** than stars (very popular repos get *relatively* fewer forks)
- slope > 1 → forks grow **faster** than stars

In [ ]:
fit_df = df[(df["forks"] > 0) & (df["stars"] > 0)]
x = np.log10(fit_df["stars"])
y = np.log10(fit_df["forks"])

slope, intercept = np.polyfit(x, y, 1)
r2 = np.corrcoef(x, y)[0, 1] ** 2
print(f"log10(forks) = {slope:.2f} × log10(stars) {intercept:+.2f}")
print(f"R² = {r2:.2f}")
print(f"\nIn plain English: 10× more stars → about {10**slope:.1f}× more forks")

plt.figure(figsize=(8, 5.5))
sns.regplot(x=x, y=y, scatter_kws={"alpha": 0.4, "s": 15, "color": BLUE},
            line_kws={"color": ORANGE})
plt.xlabel("log10(stars)")
plt.ylabel("log10(forks)")
plt.title("Stars vs forks with regression line (log-log)")
plt.show()

**What to notice:** read the slope and the plain-English line. Then look at the **outliers**: repos far *above* the line get forked a lot for their popularity (often templates, starter kits or tutorials people copy), and repos far *below* get starred but rarely forked (often apps people just use, or lists people bookmark).

### ✏️ Try it yourself

Find the 10 repos with the highest **forks-to-stars ratio** (with at least 500 stars). What kinds of projects are they?

<details><summary>Show answer</summary>

```python
df["fork_ratio"] = df["forks"] / df["stars"]
df[df["stars"] >= 500].nlargest(10, "fork_ratio")[["name", "stars", "forks", "fork_ratio", "description"]]
```
High fork ratios usually mean templates, courses, or tools people are expected to copy and customise.
</details>

In [ ]:
# Write your code here

---
# Part 7: Conclusions

## Step 7.1: Collect your key numbers

This cell gathers the main results into one summary, so your write-up uses **your** live numbers.

In [ ]:
top_lang = lang_counts.index[0]
top_lang_share = lang_counts.iloc[0] / len(df) * 100
top_topic_list = ", ".join(top_topics.head(5).index)
ai_med = ai_compare.loc["AI", "median_stars"] if "AI" in ai_compare.index else float("nan")
non_ai_med = ai_compare.loc["Not AI", "median_stars"] if "Not AI" in ai_compare.index else float("nan")

print(f"Snapshot date: {snapshot_date}   |   Repos analysed: {len(df)} (created since {since})")
print(f"Median stars: {df['stars'].median():,.0f}   |   Mean stars: {df['stars'].mean():,.0f}")
print(f"Most common language: {top_lang} ({top_lang_share:.0f}% of repos)")
print(f"Top 5 topics: {top_topic_list}")
print(f"AI-related share: {df['is_ai'].mean()*100:.0f}%")
print(f"Median stars, AI vs not AI: {ai_med:,.0f} vs {non_ai_med:,.0f}")
print(f"Fastest-growing repo: {fastest.iloc[0]['name']} ({fastest.iloc[0]['stars_per_day']:,.0f} stars/day)")
print(f"Stars vs forks: Spearman ρ = {spearman:.2f}; 10× stars → {10**slope:.1f}× forks")
print(f"Active in last 30 days: {active:.0f}%")

## Step 7.2: Write up your findings

Double-click this cell and replace the *[brackets]* with your own numbers and observations. Imagine your reader is a friend who works in tech but doesn't code in Python.

---

### What's hot in tech in 2026: findings

**Data:** the [N] most-starred GitHub repositories created between [since] and [snapshot date], collected from the GitHub search API.

1. **[Language] dominates**, making up [X]% of hot repos, followed by [language] and [language]. *[One sentence on why you think that is.]*
2. **AI is [the biggest theme / one of several themes].** About [X]% of hot repos are AI-related, and the most common topics are [topics]. AI repos have a median of [X] stars vs [Y] for non-AI repos.
3. **The fastest-growing projects right now are [examples]**, gaining up to [X] stars per day.
4. **Popularity is extremely uneven.** The median repo has [X] stars, but the mean is [Y], because a few breakout hits pull the average up.
5. **Stars and forks rise together, but not equally:** 10× more stars means about [X]× more forks. Projects people copy (like [example]) stand out above the line.

### Limitations

- GitHub search returns at most 1,000 results, so this is the **top** of the market, not all new repos.
- **Time bias:** newer repos have had less time to gain stars, so they're under-represented by count (Step 5.2).
- The AI flag uses keywords and makes some mistakes (Step 4.2).
- Stars measure attention, not quality or real-world use.
- This is a snapshot from one day. Results will change.

---

## 🎉 You're done! What next?

- **Publish it:** create a GitHub repo, upload this notebook **and** your saved CSV snapshot, and write a README with your top 3 findings and one chart. That's a strong portfolio piece, and fitting for a project *about* GitHub.
- **Repeat it next month** and compare snapshots: which repos kept growing and which faded?
- **Go further:** pick one topic (e.g. `mcp` or `rust`) and change the search query to `created:>{since} topic:mcp` to dig into just that area.